In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("msambare/fer2013")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fer2013' dataset.
Path to dataset files: /kaggle/input/fer2013


In [ ]:
import os

# Gather unique directory paths containing images
unique_dirs = set()
for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            # Get the path relative to the directory containing 'kdef-database'
            base_dir = os.path.dirname(os.path.dirname(os.path.dirname(path)))
            relative_dir = os.path.relpath(root, base_dir)
            unique_dirs.add(relative_dir)

# Print the paths (without filenames)
for d in sorted(list(unique_dirs)):
    print(f"{d}/")

kaggle/input/fer2013/test/angry/
kaggle/input/fer2013/test/disgust/
kaggle/input/fer2013/test/fear/
kaggle/input/fer2013/test/happy/
kaggle/input/fer2013/test/neutral/
kaggle/input/fer2013/test/sad/
kaggle/input/fer2013/test/surprise/
kaggle/input/fer2013/train/angry/
kaggle/input/fer2013/train/disgust/
kaggle/input/fer2013/train/fear/
kaggle/input/fer2013/train/happy/
kaggle/input/fer2013/train/neutral/
kaggle/input/fer2013/train/sad/
kaggle/input/fer2013/train/surprise/


In [ ]:
import os
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt

# 1. Gather all image files from the dataset path
image_files = []
for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            image_files.append(os.path.join(root, file))

print(f"Found {len(image_files)} images in the dataset.")

Found 35887 images in the dataset.


In [ ]:
# 2. Define the preprocessing pipeline
# Standard ImageNet normalization values are often used when resizing to 224x224
preprocess_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),        # Resize to 224x224
    transforms.Lambda(lambda x: x.convert("RGB")), # Convert to RGB (3 channels)
    transforms.ToTensor(),                # Convert to Tensor (0-1 range)
    transforms.Normalize(                 # Normalize
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 3. Apply to a sample image
if image_files:
    sample_image_path = image_files[0]
    original_image = Image.open(sample_image_path)

    # Apply preprocessing
    processed_tensor = preprocess_pipeline(original_image)

    print(f"Original Size: {original_image.size}")
    print(f"Processed Tensor Shape: {processed_tensor.shape}")
    print(f"Tensor Stats - Min: {processed_tensor.min():.3f}, Max: {processed_tensor.max():.3f}, Mean: {processed_tensor.mean():.3f}")
else:
    print("No images found to process.")

Original Size: (48, 48)
Processed Tensor Shape: torch.Size([3, 224, 224])
Tensor Stats - Min: -1.587, Max: 2.030, Mean: -0.327


In [ ]:
import os
import torch
from tqdm import tqdm
from PIL import Image

# Define output directory in Drive
output_dir = '/content/drive/MyDrive/processed_fer2013_pt'

print(f"Saving processed tensors to: {output_dir}")

# Mapping for capitalized class names
class_mapping = {
    'surprise': 'Surprise',
    'fear': 'Fear',
    'disgust': 'Disgust',
    'happy': 'Happy',
    'sad': 'Sad',
    'angry': 'Angry',
    'neutral': 'Neutral'
}

# Process and save all images
successful_saves = 0

for img_path in tqdm(image_files, desc="Processing Images"):
    try:
        # Open image
        img = Image.open(img_path)

        # Apply full preprocessing pipeline (returns a torch.Tensor)
        processed_tensor = preprocess_pipeline(img)

        # Extract the split (train/test), original class name, and filename from the path
        parts = img_path.split(os.sep)
        split_name = parts[-3] # 'train' or 'test'
        original_class = parts[-2].lower()
        filename = parts[-1]

        # Get the new capitalized class name
        new_class = class_mapping.get(original_class, original_class.capitalize())

        # Extract the ID by splitting on '_'
        file_id = os.path.splitext(filename)[0].split('_')[-1]

        # Create the new filename with .pt extension
        filename_pt = f"fer2013_{file_id}.pt"

        # Construct full save path directly under the output_dir with the split and new class name
        save_path = os.path.join(output_dir, split_name, new_class, filename_pt)

        # Ensure the subdirectory exists
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        # Save as a PyTorch tensor file
        torch.save(processed_tensor, save_path)
        successful_saves += 1
    except Exception as e:
        print(f"Error processing {img_path}: {e}")

print(f"Successfully saved {successful_saves} tensors to Google Drive.")


Saving processed tensors to: /content/drive/MyDrive/processed_fer2013_pt


Processing Images: 100%|██████████| 35887/35887 [18:23<00:00, 32.52it/s]

Successfully saved 35887 tensors to Google Drive.


In [ ]:
import os

dir_path = '/content/drive/MyDrive/processed_fer2013_pt'

if os.path.exists(dir_path):
    total_files = sum(len(files) for root, dirs, files in os.walk(dir_path))
    print(f"Total files in '{dir_path}': {total_files}")
else:
    print(f"Directory '{dir_path}' does not exist.")

Total files in '/content/drive/MyDrive/processed_fer2013_pt': 35886


In [ ]:
import os
import pandas as pd
from collections import Counter

# Path to the directory where we saved the splits (Fixed the typo from '-' to '_')
dir_path = '/content/drive/MyDrive/processed_fer2013_pt'

train_counts = Counter()
test_counts = Counter()

# Count images by split and class directly from the folders
if os.path.exists(dir_path):
    for split_name in ['train', 'test']:
        split_path = os.path.join(dir_path, split_name)
        if os.path.exists(split_path):
            for class_name in os.listdir(split_path):
                class_dir = os.path.join(split_path, class_name)
                if os.path.isdir(class_dir):
                    num_files = len([f for f in os.listdir(class_dir) if f.endswith('.pt')])
                    if split_name == 'train':
                        train_counts[class_name] += num_files
                    elif split_name == 'test':
                        test_counts[class_name] += num_files

# Get all unique classes
all_classes = sorted(list(set(train_counts.keys()).union(set(test_counts.keys()))))

# Create a DataFrame for visualization
df_counts = pd.DataFrame({
    'Class': all_classes,
    'Train Count': [train_counts.get(c, 0) for c in all_classes],
    'Test Count': [test_counts.get(c, 0) for c in all_classes]
})

# Add Total column
df_counts['Total'] = df_counts['Train Count'] + df_counts['Test Count']

# Add a Total row at the bottom
total_row = pd.DataFrame({
    'Class': ['All Classes'],
    'Train Count': [df_counts['Train Count'].sum()],
    'Test Count': [df_counts['Test Count'].sum()],
    'Total': [df_counts['Total'].sum()]
})

df_counts = pd.concat([df_counts, total_row], ignore_index=True)

# Display the results
display(df_counts)


,Class,Train Count,Test Count,Total
0,Angry,3995,958,4953
1,Disgust,436,111,547
2,Fear,4097,1024,5121
3,Happy,7215,1773,8988
4,Neutral,4965,1233,6198
5,Sad,4830,1247,6077
6,Surprise,3171,831,4002
7,All Classes,28709,7177,35886


In [ ]:
from google.colab import runtime
runtime.unassign()